# Explore taxonomy RDF — Virtuoso SPARQL queries

Queries the local Virtuoso instance via SPARQLWrapper. All queries run against named graphs, so cross-layer joins (taxonomy ↔ core ↔ properties ↔ NCBITaxon) work without limitations.

| Graph | URI |
|---|---|
| Core pathway RDF | `http://rdf-plantmetwiki.bioinformatics.nl/graph/pathways` |
| Taxonomy extra | `http://rdf-plantmetwiki.bioinformatics.nl/graph/gpml-taxonomy-extra` |
| Properties extra | `http://rdf-plantmetwiki.bioinformatics.nl/graph/gpml-properties-extra` |
| NCBITaxon ontology | `http://rdf-plantmetwiki.bioinformatics.nl/graph/ncbitaxon` |

**Kernel:** select `plantmetwiki-rdf` (register once with `python -m ipykernel install --user --name plantmetwiki-rdf`).

Once a query looks good, copy it to **[SPARQLQueries](https://github.com/pathway-lod/SPARQLQueries)**.

In [1]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

# ── Endpoint ──────────────────────────────────────────────────────────────────
# Local Virtuoso (recommended — fast, full data, all named graphs available)
ENDPOINT = "http://localhost:8890/sparql"
# Public endpoint (no local setup, but may be slower for large aggregations):
# ENDPOINT = "https://sparql-plantmetwiki.bioinformatics.nl/sparql"

# ── Named graph URIs ──────────────────────────────────────────────────────────
BASE      = "http://rdf-plantmetwiki.bioinformatics.nl"
G_CORE    = f"{BASE}/graph/pathways"
G_TAX     = f"{BASE}/graph/gpml-taxonomy-extra"
G_PROP    = f"{BASE}/graph/gpml-properties-extra"
G_NCBI    = f"{BASE}/graph/ncbitaxon"

# ── Common prefixes ───────────────────────────────────────────────────────────
PREFIXES = """
PREFIX wp:      <http://vocabularies.wikipathways.org/wp#>
PREFIX ncbi:    <http://purl.obolibrary.org/obo/NCBITaxon_>
PREFIX pmw:     <http://rdf-plantmetwiki.bioinformatics.nl/vocab/>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf:     <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX dc:      <http://purl.org/dc/elements/1.1/>
"""

# ── Helper ────────────────────────────────────────────────────────────────────
def run_query(query: str) -> pd.DataFrame:
    """Run a SPARQL SELECT against the endpoint and return a DataFrame."""
    sp = SPARQLWrapper(ENDPOINT)
    sp.setQuery(PREFIXES + query)
    sp.setReturnFormat(JSON)
    results = sp.query().convert()
    rows = results["results"]["bindings"]
    if not rows:
        return pd.DataFrame()
    cols = list(rows[0].keys())
    return pd.DataFrame(
        [{c: r[c]["value"] for c in cols if c in r} for r in rows],
        columns=cols,
    )

print(f"Endpoint: {ENDPOINT}")
print("Ready.")

Endpoint: http://localhost:8890/sparql
Ready.


---
## 1. Pathway IRIs and PlantCyc IDs

Each pathway has a stable IRI (`/pathways/PC{n}_r{version}`) and a PlantCyc ID stored as `pmw:plantcycId`.

In [2]:
# Sample pathway IRI → PlantCyc ID
run_query(f"""
SELECT ?pathway_iri ?plantcyc_id
WHERE {{
  GRAPH <{G_PROP}> {{
    ?pathway_iri pmw:plantcycId ?plantcyc_id .
    FILTER(CONTAINS(STR(?pathway_iri), "/pathways/"))
  }}
}}
LIMIT 10
""")

,pathway_iri,plantcyc_id
0,http://rdf-plantmetwiki.bioinformatics.nl/path...,ARGININE-SYN4-PWY
1,http://rdf-plantmetwiki.bioinformatics.nl/path...,GLUCONEO-PWY
2,http://rdf-plantmetwiki.bioinformatics.nl/path...,GLUTAMATE-SYN2-PWY
3,http://rdf-plantmetwiki.bioinformatics.nl/path...,GLYSYN2-PWY
4,http://rdf-plantmetwiki.bioinformatics.nl/path...,ILEUSYN-PWY
5,http://rdf-plantmetwiki.bioinformatics.nl/path...,LEU-DEG2-PWY
6,http://rdf-plantmetwiki.bioinformatics.nl/path...,LEUSYN-PWY
7,http://rdf-plantmetwiki.bioinformatics.nl/path...,NONMEVIPP-PWY
8,http://rdf-plantmetwiki.bioinformatics.nl/path...,PANTO-PWY
9,http://rdf-plantmetwiki.bioinformatics.nl/path...,PWY-1001


In [3]:
# Total pathways (PC* only — reactions use RC* identifiers)
run_query(f"""
SELECT (COUNT(DISTINCT ?iri) AS ?pathways)
WHERE {{
  GRAPH <{G_PROP}> {{
    ?iri pmw:plantcycId ?id .
    FILTER(REGEX(STR(?iri), "/pathways/PC[0-9]+_"))
  }}
}}
""")

,pathways
0,1162


---
## 2. GPML properties

All `<Property key="..." value="...">` elements are stored as blank nodes:
```turtle
?subject pmw:gpmlProperty [ pmw:key "..." ; pmw:value "..." ] .
```

In [4]:
# All unique property keys and their frequency — top 20
run_query(f"""
SELECT ?key (COUNT(?key) AS ?count)
WHERE {{
  GRAPH <{G_PROP}> {{
    ?subject pmw:gpmlProperty ?bn .
    ?bn pmw:key ?key .
  }}
}}
GROUP BY ?key
ORDER BY DESC(?count)
LIMIT 20
""")

,key,count
0,InstanceNameTemplate,54578
1,UniqueID,51528
2,Synonym_1,27711
3,Gibbs0,27049
4,Osmolarity,22564
5,Smiles,21976
6,NonStandardInchi,21563
7,MolecularWeight,20813
8,ChemicalFormula,20698
9,MonoisotopicMw,20695


In [5]:
# Original per-pathway species list from PlantCyc
# The pathway organism attribute is "Viridiplantae" in GPML, but the original
# multi-species string from PlantCyc is preserved as Property key="Organism".
run_query(f"""
SELECT ?pathway_iri ?original_species
WHERE {{
  GRAPH <{G_PROP}> {{
    ?pathway_iri pmw:gpmlProperty ?bn .
    ?bn pmw:key   "Organism" ;
        pmw:value ?original_species .
    FILTER(CONTAINS(STR(?pathway_iri), "/pathways/"))
  }}
}}
LIMIT 10
""")

,pathway_iri,original_species
0,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Brassica oleracea, Brassica juncea, Astragalus..."
1,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Chlamydomonas reinhardtii, Physcomitrium paten..."
2,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Populus trichocarpa, Arabidopsis thaliana"
3,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Lycopersicon hirsutum, Solanum, Solanum habroc..."
4,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Arabidopsis thaliana, Brassica napus, Limnanth..."
5,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Zea mays, Megathyrsus maximus, Urochloa panico..."
6,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Vitis vinifera, Zea mays, Cinnamomum tenuipile..."
7,http://rdf-plantmetwiki.bioinformatics.nl/path...,Arabidopsis thaliana
8,http://rdf-plantmetwiki.bioinformatics.nl/path...,Arabidopsis thaliana
9,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Arabidopsis thaliana, Arabidopsis thaliana, Gl..."


---
## 3. Taxonomy overview

Basic counts and species distribution from the taxonomy-extra graph.

In [6]:
# Resources annotated with Viridiplantae — expected 2478 (1162 pathways + 1316 reactions)
run_query(f"""
SELECT (COUNT(DISTINCT ?resource) AS ?resources_with_viridiplantae)
WHERE {{
  GRAPH <{G_TAX}> {{
    ?resource wp:organism ncbi:33090 .
  }}
}}
""")

,resources_with_viridiplantae
0,2478


In [7]:
# Species distribution on DataNodes — top 20, with NCBI labels (cross-graph join)
run_query(f"""
SELECT ?taxon ?label (COUNT(DISTINCT ?node) AS ?node_count)
WHERE {{
  GRAPH <{G_TAX}> {{
    ?node wp:organism ?taxon .
    FILTER(?taxon != ncbi:33090)
    FILTER(CONTAINS(STR(?node), "/DataNode/"))
  }}
  OPTIONAL {{
    GRAPH <{G_NCBI}> {{ ?taxon rdfs:label ?label . }}
  }}
}}
GROUP BY ?taxon ?label
ORDER BY DESC(?node_count)
LIMIT 20
""")

,taxon,label,node_count
0,http://purl.obolibrary.org/obo/NCBITaxon_3702,Arabidopsis thaliana,7720
1,http://purl.obolibrary.org/obo/NCBITaxon_3847,Glycine max,1078
2,http://purl.obolibrary.org/obo/NCBITaxon_34305,Lotus japonicus,926
3,http://purl.obolibrary.org/obo/NCBITaxon_3055,Chlamydomonas reinhardtii,525
4,http://purl.obolibrary.org/obo/NCBITaxon_381124,Zea mays subsp. mays,513
5,http://purl.obolibrary.org/obo/NCBITaxon_4081,Solanum lycopersicum,422
6,http://purl.obolibrary.org/obo/NCBITaxon_4113,Solanum tuberosum,305
7,http://purl.obolibrary.org/obo/NCBITaxon_3888,Lathyrus oleraceus,236
8,http://purl.obolibrary.org/obo/NCBITaxon_46611,Abies grandis,230
9,http://purl.obolibrary.org/obo/NCBITaxon_39947,Oryza sativa Japonica Group,214


In [8]:
# Total species-annotated DataNodes (excluding Viridiplantae catch-all)
run_query(f"""
SELECT (COUNT(DISTINCT ?node) AS ?annotated_datanodes)
WHERE {{
  GRAPH <{G_TAX}> {{
    ?node wp:organism ?taxon .
    FILTER(?taxon != ncbi:33090)
    FILTER(CONTAINS(STR(?node), "/DataNode/"))
  }}
}}
""")

,annotated_datanodes
0,18762


---
## 4. Cross-layer queries (taxonomy ↔ core)

In [9]:
# Pathways containing at least one Arabidopsis thaliana entity.
#
# Cross-layer join pattern:
#   G_TAX: ?entity wp:organism ncbi:3702      (entity = identifiers.org URI)
#   G_CORE: ?entity dcterms:isPartOf ?pathway  (entity → canonical pathway IRI)
#   G_CORE: ?pathway dc:title ?title
#
# (G_CORE does not use wp:isPartOf; the entity→pathway relationship uses dcterms:isPartOf)
run_query(f"""
SELECT ?pathway ?title (COUNT(DISTINCT ?entity) AS ?arabidopsis_entities)
WHERE {{
  GRAPH <{G_TAX}> {{
    ?entity wp:organism ncbi:3702 .
  }}
  GRAPH <{G_CORE}> {{
    ?entity dcterms:isPartOf ?pathway .
    ?pathway dc:title ?title .
  }}
}}
GROUP BY ?pathway ?title
ORDER BY DESC(?arabidopsis_entities)
LIMIT 20
""")

,pathway,title,arabidopsis_entities
0,http://rdf-plantmetwiki.bioinformatics.nl/path...,superpathway of flavones and derivatives biosy...,70
1,http://rdf-plantmetwiki.bioinformatics.nl/path...,"superpathway of cytosolic glycolysis (plants),...",64
2,http://rdf-plantmetwiki.bioinformatics.nl/path...,quercetin glucoside biosynthesis (Allium),62
3,http://rdf-plantmetwiki.bioinformatics.nl/path...,quercetin glycoside biosynthesis (Arabidopsis),60
4,http://rdf-plantmetwiki.bioinformatics.nl/path...,superpathway of flavones and derivatives biosy...,60
5,http://rdf-plantmetwiki.bioinformatics.nl/path...,superpathway of anaerobic sucrose degradation,59
6,http://rdf-plantmetwiki.bioinformatics.nl/path...,superpathway of phospholipid biosynthesis II (...,55
7,http://rdf-plantmetwiki.bioinformatics.nl/path...,superpathway of sucrose and starch metabolism ...,52
8,http://rdf-plantmetwiki.bioinformatics.nl/path...,rutin biosynthesis,50
9,http://rdf-plantmetwiki.bioinformatics.nl/path...,flavonol acylglucoside biosynthesis III-querce...,48


---
## 5. NCBITaxon coverage analysis

Checks which taxa present in the knowledge graph are absent from the loaded NCBITaxon ontology graph.

**Why taxa may be missing:**
- The OBO Foundry NCBITaxon release covers taxa relevant to biological ontologies — it is **not** a mirror of the full NCBI Taxonomy database.
- A taxon can also be missing because it was **deprecated and merged** into another taxon in a newer NCBI Taxonomy release.

**What is affected:**
Missing taxa can only affect GeneProduct and Protein nodes — these are the only node types that carry species-specific `wp:organism` annotations derived from PlantCyc's `proteins.dat` SPECIES field. Metabolites carry no direct species annotations.

In [10]:
# All unique taxa in the knowledge graph (excluding Viridiplantae top-level)
df_all_taxa = run_query(f"""
SELECT DISTINCT ?taxon
WHERE {{
  GRAPH <{G_TAX}> {{
    ?node wp:organism ?taxon .
    FILTER(?taxon != ncbi:33090)
  }}
}}
""")
print(f"Unique taxa in knowledge graph (excl. Viridiplantae): {len(df_all_taxa)}")

Unique taxa in knowledge graph (excl. Viridiplantae): 423


In [11]:
# Taxa present in the data but ABSENT from the NCBITaxon graph.
#
# FILTER NOT EXISTS { GRAPH ... } is unreliable in Virtuoso for cross-graph negation.
# Strategy: fetch the taxa that DO have a label, then subtract in Python.
#
# G_TAX stores wp:organism on two resource types per DataNode:
#   (a) the GPML DataNode IRI  (.../Pathway/PC.../DataNode/...)
#   (b) the biological entity   (https://identifiers.org/...)
# We count only (a) — GPML DataNode instances — to match what the text reports.

# Step 1 — taxa in data that ARE in NCBITaxon (have rdfs:label)
df_present = run_query(f"""
SELECT DISTINCT ?taxon
WHERE {{
  GRAPH <{G_TAX}> {{
    ?node wp:organism ?taxon .
    FILTER(?taxon != ncbi:33090)
  }}
  GRAPH <{G_NCBI}> {{ ?taxon rdfs:label ?label . }}
}}
""")
present_set = set(df_present["taxon"]) if not df_present.empty else set()

# Step 2 — set difference (Python; avoids FILTER NOT EXISTS cross-graph issue)
all_iris     = df_all_taxa["taxon"].tolist() if not df_all_taxa.empty else []
missing_iris = [t for t in all_iris if t not in present_set]

# Step 3 — count affected GPML DataNode instances per missing taxon
if missing_iris:
    vals = " ".join(f"(<{t}>)" for t in missing_iris)
    df_missing = run_query(f"""
SELECT ?taxon (COUNT(DISTINCT ?node) AS ?datanode_instances)
WHERE {{
  VALUES (?taxon) {{ {vals} }}
  GRAPH <{G_TAX}> {{
    ?node wp:organism ?taxon .
    FILTER(CONTAINS(STR(?node), "/DataNode/"))
  }}
}}
GROUP BY ?taxon
ORDER BY DESC(?datanode_instances)
    """)
else:
    df_missing = pd.DataFrame(columns=["taxon", "datanode_instances"])

if not df_missing.empty:
    df_missing["ncbi_id"] = df_missing["taxon"].str.extract(r'NCBITaxon_(\d+)')
    display(df_missing[["ncbi_id", "taxon", "datanode_instances"]])

print(f"Taxa absent from NCBITaxon graph : {len(df_missing)}")
if not df_missing.empty:
    print(f"Total affected DataNode instances: {df_missing['datanode_instances'].astype(int).sum()}")

,ncbi_id,taxon,datanode_instances
0,101602,http://purl.obolibrary.org/obo/NCBITaxon_101602,12
1,48038,http://purl.obolibrary.org/obo/NCBITaxon_48038,10
2,121094,http://purl.obolibrary.org/obo/NCBITaxon_121094,8
3,283673,http://purl.obolibrary.org/obo/NCBITaxon_283673,4
4,135200,http://purl.obolibrary.org/obo/NCBITaxon_135200,2
5,23810,http://purl.obolibrary.org/obo/NCBITaxon_23810,1


Taxa absent from NCBITaxon graph : 6
Total affected DataNode instances: 37


In [12]:
# DataNodes affected by missing taxa — node type + pathway title.
#
# Data model in G_CORE (GPML2021 RDF):
#   identifiers.org entity   = subject of wp:isAbout, dcterms:isPartOf, rdf:type
#   GPML DataNode IRI        = object of wp:isAbout (from G_TAX)
#
# So: ?entity wp:isAbout ?node   (not the reverse)
#     ?entity rdf:type ?node_type   (not ?node rdf:type)
#     ?entity dcterms:isPartOf ?pathway
#     ?pathway dc:title ?pathway_title

if df_missing.empty:
    print("No missing taxa — nothing to report.")
    df_affected = pd.DataFrame(columns=["taxon", "node", "entity", "node_type", "pathway", "pathway_title"])
else:
    vals = " ".join(f"(<{t}>)" for t in missing_iris)
    df_affected = run_query(f"""
SELECT ?taxon ?node ?entity ?node_type ?pathway ?pathway_title
WHERE {{
  VALUES (?taxon) {{ {vals} }}
  GRAPH <{G_TAX}> {{
    ?node wp:organism ?taxon .
    FILTER(CONTAINS(STR(?node), "/DataNode/"))
  }}
  GRAPH <{G_CORE}> {{
    ?entity wp:isAbout ?node .
    ?entity rdf:type ?node_type .
    ?entity dcterms:isPartOf ?pathway .
    ?pathway dc:title ?pathway_title .
    FILTER(STRSTARTS(STR(?node_type), "http://vocabularies.wikipathways.org/wp#"))
    FILTER(?node_type != <http://vocabularies.wikipathways.org/wp#DataNode>)
  }}
}}
ORDER BY ?taxon ?pathway_title
    """)

if not df_affected.empty:
    df_affected["ncbi_id"]    = df_affected["taxon"].str.extract(r'NCBITaxon_(\d+)')
    df_affected["type_short"] = df_affected["node_type"].str.split("#").str[-1]
    # Unique DataNode IRIs per type (deduplicate on node+type; avoids counting
    # the same DataNode multiple times just because it appears in multiple pathways)
    df_unique_types = df_affected.drop_duplicates(subset=["node", "node_type"])
    print(f"Unique DataNode instances (from df_missing) : {df_missing['datanode_instances'].astype(int).sum()}")
    print(f"(entity, pathway, type) rows in df_affected : {len(df_affected)}")
    print()
    print("Node type breakdown (per unique DataNode IRI):")
    display(df_unique_types["type_short"].value_counts().to_frame())
    print()
    print("Per-taxon, per-type breakdown:")
    display(df_affected[["ncbi_id", "type_short", "pathway_title"]].value_counts(["ncbi_id", "type_short"]).to_frame())
else:
    print("No affected DataNodes found.")

Unique DataNode instances (from df_missing) : 37
(entity, pathway, type) rows in df_affected : 85

Node type breakdown (per unique DataNode IRI):


,count
type_short,
Protein,27
GeneProduct,10



Per-taxon, per-type breakdown:


count
ncbi_id type_short        
48038   Protein         26
101602  Protein         23
        GeneProduct      9
121094  Protein          8
        GeneProduct      8
283673  Protein          4
        GeneProduct      4
135200  Protein          1
        GeneProduct      1
23810   Protein          1

In [13]:
# Summary: impact as % of all species-annotated DataNodes
df_total = run_query(f"""
SELECT (COUNT(DISTINCT ?node) AS ?total_annotated)
WHERE {{
  GRAPH <{G_TAX}> {{
    ?node wp:organism ?taxon .
    FILTER(?taxon != ncbi:33090)
    FILTER(CONTAINS(STR(?node), "/DataNode/"))
  }}
}}
""")

total          = int(df_total["total_annotated"].iloc[0])
n_missing_taxa = len(df_missing)
n_affected     = len(df_affected) if not df_affected.empty else 0
pct            = n_affected / total * 100 if total else 0

print(f"Total species-annotated DataNodes : {total:,}")
print(f"Taxa absent from NCBITaxon graph  : {n_missing_taxa}")
print(f"Affected DataNodes                : {n_affected}  ({pct:.1f}% of annotated nodes)")

if not df_affected.empty:
    print()
    print("Node types affected:")
    print(df_affected["type_short"].value_counts().to_string())

Total species-annotated DataNodes : 18,762
Taxa absent from NCBITaxon graph  : 6
Affected DataNodes                : 85  (0.5% of annotated nodes)

Node types affected:
type_short
Protein        63
GeneProduct    22


In [14]:
# Look up species names for the 6 missing taxa from the GPML Annotation elements.
# Format in GPML: <Annotation elementId="taxonomy_{ncbi_id}" value="Species name" type="Taxonomy">
# The elementId and value are on the same line, so grep on elementId suffix gives the name.
import subprocess, re

gpml_dir = "../input/gpml/renamed/pathways"

taxon_names = {}
for iri in missing_iris:
    ncbi_id = iri.split("NCBITaxon_")[-1]
    try:
        result = subprocess.run(
            ["grep", "-r", f"taxonomy_{ncbi_id}", gpml_dir],
            capture_output=True, text=True, timeout=30
        )
        # Line format: <Annotation elementId="taxonomy_101602" value="Dahlia variabilis" type="Taxonomy">
        names = set(re.findall(r'value="([^"]+)"', result.stdout))
        taxon_names[ncbi_id] = ", ".join(sorted(names)) if names else "(not found in GPML)"
    except Exception as e:
        taxon_names[ncbi_id] = f"(error: {e})"

print("Taxon names from GPML Annotation elements:")
for ncbi_id, name in taxon_names.items():
    print(f"  NCBITaxon_{ncbi_id}: {name}")

Taxon names from GPML Annotation elements:
  NCBITaxon_101602: Dahlia variabilis
  NCBITaxon_121094: Perilla citriodora
  NCBITaxon_135200: Schizonepeta tenuifolia
  NCBITaxon_23810: Ailanthus altissima
  NCBITaxon_283673: Lycopersicon hirsutum
  NCBITaxon_48038: Foeniculum vulgare


In [15]:
# ── Auto-generated reflection paragraph ───────────────────────────────────────
# Reproduces the NCBITaxon coverage paragraph for the manuscript/documentation,
# updated with numbers from the current Virtuoso instance.
# Re-run after any data update.

# NCBITaxon graph stats
df_ncbi_stats = run_query(f"""
SELECT (COUNT(*) AS ?triples)
WHERE {{ GRAPH <{G_NCBI}> {{ ?s ?p ?o }} }}
""")
ncbi_triples = int(df_ncbi_stats["triples"].iloc[0])

# NCBITaxon release date from owl:versionInfo
df_ncbi_date = run_query(f"""
SELECT ?v
WHERE {{
  GRAPH <{G_NCBI}> {{
    ?s <http://www.w3.org/2002/07/owl#versionInfo> ?v .
  }}
}}
LIMIT 1
""")
ncbi_release = df_ncbi_date["v"].iloc[0] if not df_ncbi_date.empty else "unknown"

total_ann   = int(df_total["total_annotated"].iloc[0])
n_missing   = len(df_missing)
# Use unique DataNode IRIs (df_missing) as the authoritative "nodes affected" count.
n_nodes     = df_missing["datanode_instances"].astype(int).sum() if not df_missing.empty else 0
pct         = n_nodes / total_ann * 100 if total_ann else 0

# Node types from deduplicated df_affected (unique DataNode IRI × type)
if not df_affected.empty:
    type_counts = df_affected.drop_duplicates(subset=["node", "node_type"])["type_short"].value_counts().to_dict()
else:
    type_counts = {}
gp_count  = type_counts.get("GeneProduct", 0)
pr_count  = type_counts.get("Protein", 0)
met_count = type_counts.get("Metabolite", 0)

if gp_count and pr_count:
    type_str = "GeneProduct and Protein"
elif gp_count:
    type_str = "GeneProduct"
elif pr_count:
    type_str = "Protein"
else:
    type_str = "gene or enzyme"

# Classify: deprecated vs valid-but-not-in-OBO
deprecated = {k: v for k, v in taxon_names.items() if "hirsutum" in v.lower() or "Lycopersicon" in v}
not_in_obo = {k: v for k, v in taxon_names.items() if k not in deprecated}

depr_str = ""
if deprecated:
    dep_name, dep_id = list(deprecated.values())[0], list(deprecated.keys())[0]
    others = ", ".join(v for k, v in not_in_obo.items())
    depr_str = (
        f"One taxon ({dep_name}, NCBI:{dep_id}) has been deprecated and merged "
        f"into Solanum habrochaites (NCBI:62890) in current NCBI Taxonomy releases, "
        f"and was therefore removed from the OBO Foundry OWL file; "
        f"the remaining {n_missing - 1} "
        f"({others}) are valid NCBI taxa that are not included in the OBO Foundry release, "
        f"which covers biologically relevant taxa used in ontology annotation rather than "
        f"the complete NCBI Taxonomy database."
    )

para = (
    f"When loading the full OBO Foundry NCBITaxon release "
    f"(v{ncbi_release}, {ncbi_triples:,} triples), "
    f"{n_missing} taxa present in PlantCyc 17.0 were absent from the ontology, "
    f"totalling {n_nodes} {type_str} nodes from the modelled pathways that could not "
    f"be resolved to a taxon label. "
    f"All affected nodes are gene or enzyme annotations — no metabolite nodes are affected. "
    f"{depr_str} "
    f"These {n_nodes} nodes represent {pct:.1f}% of the {total_ann:,} "
    f"species-annotated DataNodes in the knowledge graph and do not affect "
    f"pathway structure or metabolite annotations."
)

print("=" * 72)
print("REFLECTION PARAGRAPH (copy to manuscript/documentation)")
print("=" * 72)
print()
print(para)

REFLECTION PARAGRAPH (copy to manuscript/documentation)

When loading the full OBO Foundry NCBITaxon release (v2026-05-13, 21,719,262 triples), 6 taxa present in PlantCyc 17.0 were absent from the ontology, totalling 37 GeneProduct and Protein nodes from the modelled pathways that could not be resolved to a taxon label. All affected nodes are gene or enzyme annotations — no metabolite nodes are affected. One taxon (Lycopersicon hirsutum, NCBI:283673) has been deprecated and merged into Solanum habrochaites (NCBI:62890) in current NCBI Taxonomy releases, and was therefore removed from the OBO Foundry OWL file; the remaining 5 (Dahlia variabilis, Perilla citriodora, Schizonepeta tenuifolia, Ailanthus altissima, Foeniculum vulgare) are valid NCBI taxa that are not included in the OBO Foundry release, which covers biologically relevant taxa used in ontology annotation rather than the complete NCBI Taxonomy database. These 37 nodes represent 0.2% of the 18,762 species-annotated DataNodes in

---
## 6. Sandbox

Write new queries here. Use named graphs explicitly. When a query is validated, add it to **[SPARQLQueries](https://github.com/pathway-lod/SPARQLQueries)**.

In [16]:
run_query(f"""
SELECT *
WHERE {{
  # ← write your query here
}}
LIMIT 20
""")

,_star_fake
0,1
